In [1]:
# ========================================
# Library Imports
# ========================================
import numpy as np
import pandas as pd
import selfies as sf

In [6]:
# ========================================
# Dataset Building Helpers
# ========================================
def smiles_report(seq_data_df):
    print("smiles shape:", seq_data_df.shape)
    print("unique smiles:", seq_data_df['smiles'].nunique())
    print("smiles with string value na:", seq_data_df['smiles'].isna().sum())
    print("min, max lengths of smiles strings:", 
          seq_data_df['smiles'].astype(str).str.len().min(), 
          seq_data_df['smiles'].astype(str).str.len().max())
    print()
    
def selfies_report(seq_data_df):
    print("selfies shape:", seq_data_df.shape)
    print("unique selfies:", seq_data_df['selfies'].nunique())
    print("selfies with string value na:", seq_data_df['selfies'].isna().sum())
    print("min, max lengths of selfies strings:", 
          seq_data_df['selfies'].astype(str).str.len().min(), 
          seq_data_df['selfies'].astype(str).str.len().max())
    print()

def safe_apply_selfies_encoder(smiles):
    '''This function handles the exception the selfies encoder throws 
    for an invalid SMILE string.  This makes the function safe to use as 
    input to the pandas dataframe apply() function.  
    '''
    try:
        return sf.encoder(smiles)
    except Exception:
        # Return NaN if any exception occurs
        return np.nan 

In [12]:
# ================================================================================
# Dataset of Known Inhibitors (from literature), "650" mentioned in the paper
# ================================================================================
known_inhibitors = "data/KRAS_G12D_inhibitors_update202209_updated.csv"
df_known_inhibs = pd.read_csv(known_inhibitors)
# Check the data
smiles_report(df_known_inhibs)
# Add SELFIES to the df, by calling the encoder
df_known_inhibs['selfies'] = df_known_inhibs['smiles'].apply(safe_apply_selfies_encoder)
# Delete rows where the encoder found an invalid SMILES string (result NaN)
df_known_inhibs = df_known_inhibs.dropna(subset=['selfies'])
# Check the data
selfies_report(df_known_inhibs)
smiles_report(df_known_inhibs)
df_known_inhibs.head()
# --------------------------------------------------------------------------------------------------
# The reports imply: 
# three of the SMILES were invalid, so we dropped those
# there are six duplicate SMILES and their SELFIES in the data set, and we haven't dropped those
# --------------------------------------------------------------------------------------------------

smiles shape: (645, 5)
unique smiles: 639
smiles with string value na: 0
min, max lengths of smiles strings: 51 109

selfies shape: (636, 6)
unique selfies: 630
selfies with string value na: 0
min, max lengths of selfies strings: 252 518

smiles shape: (636, 6)
unique smiles: 630
smiles with string value na: 0
min, max lengths of smiles strings: 51 109



,id,smiles,KRAS G12D SPR KD (nM),KRAS G12D binding IC50 (nM),pERK IC50 (nM),selfies
0,1,CN1[C@H](COc2nc3cc(-c4cc(O)cc5ccccc45)ncc3c(N3...,97.7,124.7,3159.1,[C][N][C@H1][Branch2][Branch1][=Branch1][C][O]...
1,2,CN1[C@H](COc2nc(N3CC(CC4)NC4C3)c(cnc(-c3cc(O)c...,2.4,2.7,721.4,[C][N][C@H1][Branch2][Branch1][#Branch2][C][O]...
2,3,Cn1c(CCOc2nc(N3CC(CC4)NC4C3)c(cnc(-c3cc(O)cc4c...,8.3,9.5,10283.1,[C][N][C][Branch2][Branch1][O][C][C][O][C][=N]...
3,4,Oc1cc2ccccc2c(-c(ncc(c2n3)c(N4CC(CC5)NC5C4)nc3...,155.7,496.2,8530,[O][C][=C][C][=C][C][=C][C][=C][Ring1][=Branch...
4,5,Cn1nccc1COc1nc(N2CC(CC3)NC3C2)c(cnc(-c2cc(O)cc...,294.8,722.9,8193.8,[C][N][N][=C][C][=C][Ring1][Branch1][C][O][C][...


In [14]:
# =======================================================
# Full dataset of SMILES strings, 1M+
# =======================================================
# These steps took no more than 17 mins on my desktop PC
paper_full_data_set = "data/1Mstoned_vsc_initial_dataset_insilico_chemistry42_filtered.csv"
df_seqs_all = pd.read_csv(paper_full_data_set)
# Check the data
smiles_report(df_seqs_all)
# Add SELFIES to the df, by calling the encoder
df_seqs_all['selfies'] = df_seqs_all['smiles'].apply(safe_apply_selfies_encoder)
# Delete rows where the encoder found an invalid SMILES string (result NaN)
df_seqs_all = df_seqs_all.dropna(subset=['selfies'])
# Check the data
selfies_report(df_seqs_all)
smiles_report(df_seqs_all)
df_seqs_all.head()

smiles shape: (1010510, 2)
unique smiles: 1010510
smiles with string value na: 0
min, max lengths of smiles strings: 22 109

selfies shape: (1010272, 3)
unique selfies: 1010271
selfies with string value na: 0
min, max lengths of selfies strings: 81 518

smiles shape: (1010272, 3)
unique smiles: 1010272
smiles with string value na: 0
min, max lengths of smiles strings: 22 109



,Unnamed: 0,smiles,selfies
0,0,CC1=C(CNc2cccnc2C)CN(C)C1OCCCCCN,[C][C][=C][Branch1][N][C][N][C][=C][C][=C][N][...
1,1,CCN1CCCC1Cc1nc2c(c(N(C)CCCNC)n1)CCN(c1cncc3ccc...,[C][C][N][C][C][C][C][Ring1][Branch1][C][C][=N...
2,2,C1=CCC(CCOc2nc3c([nH]2)CCN(c2ccccc2)C3)CC1,[C][=C][C][C][Branch2][Ring1][=N][C][C][O][C][...
3,3,C=CC1(Oc2nc(N3CCN(CC)CC3)c3cnc(-c4ccccc4NC)c(F...,[C][=C][C][Branch2][Ring2][=C][O][C][=N][C][Br...
4,4,C=CNC(C)CCOc1nc(CN2CCNCC2)c2cnc(-c3ccccc3C(F)(...,[C][=C][N][C][Branch1][C][C][C][C][O][C][=N][C...


In [15]:
# ==================================================================
# Save the datasets containing both the SMILES and SELFIES strings
# ==================================================================
# Save the known inhibitors dataset with the SMILES and SELFIES strings, ~(636, 6)
df_known_inhibs.to_csv("data/650known_inhibitors_smiles_and_selfies.csv", index=False)
# Save the Full dataset with the SMILES and SELFIES strings, 1M+
df_seqs_all.to_csv("data/1Mstoned_vsc_initial_dataset_insilico_chemistry42_filtered_smiles_and_selfies.csv", index=False)